# Cost-aware Multi-Fidelity BO：評価費用を考慮したベイズ最適化

このNotebookでは、**評価精度（fidelity）と評価コストを同時に考慮して次の実験を選ぶ**方法を学びます。

1. 低Fidelityと目標Fidelityの違いを確認する
2. `SingleTaskMultiFidelityGP.fit()` で学習する
3. 既存の `qMultiFidelityKnowledgeGradient` で次の実験条件とFidelityを提案する
4. `CostBudget` で費用を予約・精算する
5. 実験結果を追加し、GPを再学習して次を提案する
6. **同じ追加評価予算**でMF-KGとランダム探索を比較する

**注意：** 最後のベンチマークに出てくるtarget-fidelity oracleは合成関数でのみ利用できる**診断用の真値**です。GPの学習には使わず、実運用で無料の高Fidelity評価ができるとは仮定しません。

## 1. Fidelityとは何か

ここでは入力を `X=[design, fidelity]` とします。`design` は探索する設計変数、`fidelity` は評価の忠実度です。`fidelity=1.0` を最終的に重視する高Fidelityとし、`fidelity=0.5` は比較的低コストの近似評価です。

低Fidelityは安価でも、高Fidelityの目的関数と完全に一致するとは限りません。MF-GPは**Fidelity間の関係を学習**し、どのFidelityを次に測定する価値があるか判断します。

In [ ]:
import torch
import matplotlib.pyplot as plt
from botorch.acquisition.analytic import PosteriorMean
from botorch.acquisition.cost_aware import InverseCostWeightedUtility
from botorch.acquisition.knowledge_gradient import qMultiFidelityKnowledgeGradient
from botorch.models.cost import AffineFidelityCostModel
from botorch.optim import optimize_acqf

from robotorchan.models import SingleTaskMultiFidelityGP
from robotorchan.workflow import CostBudget

torch.set_default_dtype(torch.double)
torch.manual_seed(0)

def objective(X: torch.Tensor) -> torch.Tensor:
    return torch.sin(5.0 * X[..., :1]) + 0.2 * (1.0 - X[..., 1:])

def project_to_target(X: torch.Tensor) -> torch.Tensor:
    projected = X.clone()
    projected[..., 1] = 1.0
    return projected

bounds = torch.tensor([[0.0, 0.5], [1.0, 1.0]])
target_bounds = torch.tensor([[0.0, 1.0], [1.0, 1.0]])
cost_model = AffineFidelityCostModel(fidelity_weights={1: 1.0}, fixed_cost=0.1)

## 2. Fidelityごとの応答と費用

この合成問題では `f(x,s)=sin(5x)+0.2(1-s)` とし、Fidelityが低いと応答に系統的なずれがあります。**これは例題のために既知の式を使っているだけで、実際のBOでは未知**です。

費用モデルは `cost(s)=0.1+s` です。低Fidelity `s=0.5` は0.6、高Fidelity `s=1.0` は1.1の費用です。単純に安い評価を増やせばよいのではなく、**高Fidelityの最適設計を学ぶための情報価値**も考慮します。

In [ ]:
design_grid = torch.linspace(0.0, 1.0, 201)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for fidelity in (0.5, 0.75, 1.0):
    X = torch.stack((design_grid, torch.full_like(design_grid, fidelity)), dim=-1)
    axes[0].plot(design_grid, objective(X).squeeze(-1), label=f"fidelity={fidelity}")
axes[0].set(xlabel="Design x", ylabel="Response", title="Response at each fidelity")
axes[0].legend()
fidelities = torch.linspace(0.5, 1.0, 51)
axes[1].plot(fidelities, 0.1 + fidelities)
axes[1].set(xlabel="Fidelity", ylabel="Evaluation cost", title="Affine fidelity cost")
plt.show()

## 3. 初期観測データからMF-GPを学習する

初期データとして、同じ5つの設計条件を低Fidelityと高Fidelityで評価した10件を用意します。`data_fidelities=[1]` は**2番目の入力列がFidelity**であることを意味します。

モデルの学習には既存の `fit()` を使用します。`project_to_target()` は、設計変数を維持したままFidelityだけを1.0へ変更する関数です。

In [ ]:
design = torch.linspace(0.0, 1.0, 5)
train_X = torch.cat((
    torch.stack((design, torch.full_like(design, 0.5)), dim=-1),
    torch.stack((design, torch.ones_like(design)), dim=-1),
))
train_Y = objective(train_X)
model = SingleTaskMultiFidelityGP(train_X, train_Y, data_fidelities=[1])
model.fit()
model.eval()
print("Training X:", train_X.shape, "Y:", train_Y.shape)
print("Fidelity values:", torch.unique(train_X[:, 1]))

## 4. 学習済みGPで高Fidelityの予測を確認する

最終的に知りたいのは、Fidelityを1.0に固定したときの応答です。`model.posterior()` に高Fidelityの入力を渡して、予測平均と不確実性を可視化します。

GP予測の最大値と、実際に観測した最良値は異なります。予測値は**次の実験を決めるための推定**です。

In [ ]:
target_grid = torch.stack((design_grid, torch.ones_like(design_grid)), dim=-1)
with torch.no_grad():
    posterior = model.posterior(target_grid)
    mean = posterior.mean.squeeze(-1)
    sd = posterior.variance.clamp_min(0).sqrt().squeeze(-1)
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(design_grid, objective(target_grid).squeeze(-1), label="True target response (demo only)")
ax.plot(design_grid, mean, label="MF-GP mean")
ax.fill_between(design_grid, mean-2*sd, mean+2*sd, alpha=0.2, label="Mean ± 2 SD")
ax.scatter(train_X[5:, 0], train_Y[5:, 0], s=35, label="Observed target fidelity")
ax.set(xlabel="Design x", ylabel="Target-fidelity response", title="Target fidelity posterior")
ax.legend()
plt.show()

## 5. MF-KGで次の設計条件とFidelityを提案する

`qMultiFidelityKnowledgeGradient` は、**次の測定で得られる情報が高Fidelityの最適設計の判断をどの程度改善するか**を評価します。

`current_value` は高Fidelityでの `PosteriorMean` を最大化して求めます。`InverseCostWeightedUtility` で費用を考慮し、`project` で高Fidelityへ投影します。

最適化結果の `candidate` は `[design, fidelity]` の1行です。`current_value` や獲得関数値は実測応答ではありません。

In [ ]:
_, current_value = optimize_acqf(
    PosteriorMean(model), bounds=target_bounds, q=1,
    num_restarts=2, raw_samples=8, fixed_features={1: 1.0},
)
acquisition = qMultiFidelityKnowledgeGradient(
    model=model,
    num_fantasies=4,
    current_value=current_value,
    cost_aware_utility=InverseCostWeightedUtility(cost_model=cost_model),
    project=project_to_target,
)
candidate, acquisition_value = optimize_acqf(
    acquisition, bounds=bounds, q=1, num_restarts=2, raw_samples=16,
)
candidate = candidate.detach()
predicted_cost = float(cost_model(candidate).sum().item())
print("Candidate [design, fidelity]:", candidate)
print("Predicted cost:", predicted_cost)
print("MF-KG acquisition value:", acquisition_value.detach())
assert candidate.shape == (1, 2)

## 6. 費用の予約・実測費用の精算・観測登録

`CostBudget` は費用の管理を担います。候補を評価する前に `reserve()` で予測費用を予約し、評価後に `settle()` で実測費用を精算します。

この合成例では実測費用も `0.1+fidelity` として計算します。実務では請求費用・装置時間などを使います。**費用は目的関数の応答値とは別**です。

ここでの予算は**追加評価分のみ**です。初期10観測の費用は含めていません。

In [ ]:
budget = CostBudget(total=3.0)
assert budget.can_afford(predicted_cost)
budget.reserve("evaluation-1", predicted_cost)
observed_Y = objective(candidate)
measured_cost = float((0.1 + candidate[:, 1]).sum().item())
budget.settle("evaluation-1", measured_cost)
updated_X = torch.cat((train_X, candidate), dim=0)
updated_Y = torch.cat((train_Y, observed_Y), dim=0)
print("Observed response:", observed_Y.item())
print("Spent:", budget.spent, "Remaining:", budget.remaining)
assert updated_X.shape[0] == train_X.shape[0] + 1

## 7. 観測を追加してGPを再学習し、次の候補を提案する

1件の観測が完了したら、その `[design,fidelity]` と応答を訓練データへ追加します。新しいGPを `fit()` で学習し直し、**現在のデータに基づくMF-KG** を再構築します。

次の候補を提案しただけでは予算は消費しません。実際に測定する前に `budget.can_afford()` で費用を確認します。

In [ ]:
updated_model = SingleTaskMultiFidelityGP(updated_X, updated_Y, data_fidelities=[1])
updated_model.fit()
updated_model.eval()
_, updated_current_value = optimize_acqf(
    PosteriorMean(updated_model), bounds=target_bounds, q=1,
    num_restarts=2, raw_samples=8, fixed_features={1: 1.0},
)
updated_acquisition = qMultiFidelityKnowledgeGradient(
    model=updated_model, num_fantasies=4, current_value=updated_current_value,
    cost_aware_utility=InverseCostWeightedUtility(cost_model=cost_model),
    project=project_to_target,
)
next_candidate, next_value = optimize_acqf(
    updated_acquisition, bounds=bounds, q=1, num_restarts=2, raw_samples=16,
)
next_candidate = next_candidate.detach()
next_cost = float(cost_model(next_candidate).sum().item())
print("Next [design, fidelity]:", next_candidate)
print("Next predicted cost:", next_cost)
print("Affordable with remaining budget:", budget.can_afford(next_cost))

## 8. 既存ベンチマークでMF-KGとランダム探索を比較する

ここからは既存の `benchmarks/multifidelity_cost_budget.py` を読み込み、**同じ追加評価費用の予算3.0**でランダム探索とMF-KGを比較します。

比較条件：初期観測は共通、初期観測費用は予算外、候補選択の計算時間も費用に含まない、ランダム探索も連続Fidelityから候補を選ぶ、各手法2 seedの小規模例です。

**診断用の `best_target_oracle` は合成目的関数を高Fidelityで評価した値**であり、学習データにも費用予算にも含めません。実運用では無料で取得できません。

In [ ]:
from pathlib import Path
import importlib.util
import pandas as pd

candidates = [
    Path.cwd() / "benchmarks" / "multifidelity_cost_budget.py",
    Path.cwd().parent / "benchmarks" / "multifidelity_cost_budget.py",
    Path.cwd().parent.parent / "benchmarks" / "multifidelity_cost_budget.py",
]
benchmark = next((p for p in candidates if p.is_file()), None)
if benchmark is None:
    raise FileNotFoundError("Run this notebook from the repository root or examples/notebooks directory")
spec = importlib.util.spec_from_file_location("mf_cost_benchmark", benchmark)
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)
print("Benchmark module:", benchmark)

In [ ]:
records = [row for seed in (0, 1) for policy in ("random", "mfkg")
           for row in module.experiment(policy, seed, total_budget=3.0)]
results = pd.DataFrame(records)
display(results)

## 9. 同じ費用軸で最良値の推移を比較する

横軸は試行回数ではなく**累積追加評価費用**です。Fidelityが異なる評価では、同じ試行回数でも総費用が異なります。

縦軸は合成ベンチマークの診断指標 `best_target_oracle` です。**高Fidelityの未実測値をGPに渡しているわけではありません**。少数seedの結果からアルゴリズムの優劣を断定しないでください。

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for (policy, seed), group in results.groupby(["policy", "seed"]):
    ax.plot(group["spent"], group["best_target_oracle"], marker="o", label=f"{policy}, seed={seed}")
ax.set(xlabel="Cumulative evaluation cost", ylabel="Best target-fidelity oracle value", title="Equal-cost-budget benchmark")
ax.legend()
plt.show()

## 10. Fidelityの選択と予算消費を確認する

MF-KGは、安い低Fidelity測定と高Fidelity測定の情報価値・費用を考慮します。どのFidelityを選んだかを表示し、同じ予算で何件測定できたかを確認します。

結果は合成目的関数と乱数seedに依存します。特定のFidelityを必ず選ぶという保証はありません。

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for (policy, seed), group in results.groupby(["policy", "seed"]):
    ax.plot(group["spent"], group["fidelity"], marker="o", label=f"{policy}, seed={seed}")
ax.set(xlabel="Cumulative evaluation cost", ylabel="Chosen fidelity", ylim=(0.45, 1.05), title="Fidelity choices by budget")
ax.legend()
plt.show()
print(results.groupby(["policy", "seed"]).agg(evaluations=("step", "count"), spent=("spent", "max")))

## 11. 結果の読み方と実務上の注意

- **高Fidelityの最良値を知ること**と**低Fidelityで多く測定すること**は同じ目的ではありません。
- `CostBudget` の `reserve/settle` は実験の費用管理です。獲得関数値や目的関数値を予算へ登録しません。
- 初期観測費用・計算時間はこの比較の予算に含めていません。実際の総コストを比較する場合は含める必要があります。
- target-fidelity oracleは合成ベンチマーク専用です。実運用では未知の真値を無料で利用できません。
- 2 seedの図は**挙動を学ぶための例**であり、統計的な性能優位を示すものではありません。

### 公開APIの役割

| 目的 | API |
|---|---|
| Fidelity付きGP | `SingleTaskMultiFidelityGP` |
| GP学習 | `model.fit()` |
| 高Fidelityへ投影 | `project_to_target()` |
| 目標Fidelityの現時点の推定 | `PosteriorMean` |
| 情報価値の獲得関数 | `qMultiFidelityKnowledgeGradient` |
| 評価コストのモデル | `AffineFidelityCostModel` |
| コストを考慮した獲得関数の効用 | `InverseCostWeightedUtility` |
| 候補最適化 | BoTorch `optimize_acqf` |
| 予算予約・精算 | `CostBudget` |

**次のステップ：** 合成の `objective()` と費用式を、実験装置・シミュレータ・製造評価の実測値と実費用に置き換えてください。